# LoRA SFT of Qwen3.8-Flash-Next (NVFP4) on Colab

Runtime: **G4 GPU** (RTX PRO 6000, 96 GB) with **High-RAM on**. About 8.9 compute units per hour.

Before the first run, upload `sarbloh_alignment.zip` to your Google Drive at `MyDrive/sarbloh/`.

Run order:
1. Cells 1 to 6: set-up. The model download is about 135 GB to local disk, and it is lost when the session ends.
2. Cell 7 (probe, about 15 min): peak VRAM and speed at the PROBE_LENGTHS. Send the output before training.
3. Cell 8 (train): the checkpoint and the adapter go to Drive, so after a disconnect run cells 1-6 again, then cell 8;
   it resumes from the last checkpoint. A crash restarts by itself (up to RETRIES times), also from the checkpoint.
4. Cell 9 (optional): writes the adapter into the 4 bf16 shards (about 15 GB to Drive).

In [ ]:
# 1. knobs: everything you might change is here
DRIVE_DIR = "/content/drive/MyDrive/sarbloh"          # sarbloh_alignment.zip lives here; outputs go to sft_out/
CODE_DIR = "/content/sarbloh"                          # the zip is unpacked here (local disk)
MODEL_DIR = "/content/models/flash-next-nvfp4"
MODEL_REPO = "RadixArk/Qwen3.8-Flash-Next-NVFP4"
MODEL_REV = "7b719225242aacd3dbd3f9407468c2ee9a9d2594"  # the checkpoint Kaggle serves
OFFLOAD_EXPERT_LAYERS = 36   # MoE layers whose experts stay in CPU RAM (~1.4 GB VRAM each); raise if the probe OOMs
PROBE_LENGTHS = "24576,32768"  # stops at the first OOM; MAX_SEQ_LEN must be a length that printed "ok": true
MAX_SEQ_LEN = 32768          # rows are ~75k tokens: each is cut into windows of this size (opening + recent history)
DATA = "data/*.jsonl"        # the 2026-10-20 9 h run (125/183 levels); cleared, non-held-out rows only
HELDOUT = ["ar25", "cd82", "ft09", "g50t", "m0r0", "s5i5", "sk48", "vc33"]   # never trained on
EPOCHS = 1                   # 758 windows of 32k = 24M tokens: ~66 h at 100 tok/s, so MAX_STEPS cuts it short
MAX_STEPS = 45               # optimizer steps (4 windows each) = MAX_TRAIN_HOURS*3600 / (4 * probe seconds per step)
MAX_TRAIN_HOURS = 18         # clean stop + save; 200 units / 8.9 per hour = ~22 h in total incl. set-up and probe
SAVE_EVERY = 2               # optimizer steps between checkpoints on Drive
RETRIES = 3                  # cell 8 restarts training this many times after a crash
OUT_DIR = f"{DRIVE_DIR}/sft_out/flash_next_nvfp4"

In [ ]:
# 2. machine check
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!free -g | head -2
!df -h /content | tail -1
!python --version

In [ ]:
# 3. Drive + code
from google.colab import drive
drive.mount("/content/drive")
import os
os.makedirs(OUT_DIR, exist_ok=True)
!rm -rf {CODE_DIR} && mkdir -p {CODE_DIR} && unzip -q -o {DRIVE_DIR}/sarbloh_alignment.zip -d {CODE_DIR}
!ls {CODE_DIR}/Alignment {CODE_DIR}/Alignment/data

In [ ]:
# 4. packages (training runs in a subprocess, so the notebook's own torch does not matter)
!pip install -q --index-url https://download.pytorch.org/whl/cu128 "torch==2.11.0"
!pip install -q "transformers==5.19.0" "peft==0.21.2" "accelerate==1.15.0" "safetensors==0.8.0" \
    "huggingface_hub[hf_xet]==2.2.0" "flash-linear-attention==0.5.2" matplotlib jinja2
!python -c "import torch, transformers, peft; print(torch.__version__, torch.cuda.is_available(), transformers.__version__, peft.__version__)"

In [ ]:
# 5. model download (~135 GB, resumable: rerunning skips files already there)
import os
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"
try:                                   # optional: an HF_TOKEN in Colab secrets (key icon on the left)
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass
!hf download {MODEL_REPO} --revision {MODEL_REV} --local-dir {MODEL_DIR}
!du -sh {MODEL_DIR}
!df -h /content | tail -1

In [ ]:
# 6. run config + unit test on this GPU (tiny random model, ~2 min; must end with HARD FAILS: 0)
import json
cfg = {
    "model": MODEL_DIR, "weights_format": "nvfp4", "dtype": "bf16",
    "device_map": "split", "cpu_modules": ["ngram_embedding"], "offload_expert_layers": OFFLOAD_EXPERT_LAYERS,
    "long_context": "on", "long_query_chunk": 1024, "long_moe_chunk": 32768,
    "allow_missing_weights": ["(^|\\.)mtp\\.", "lm_head"],
    "data_path": DATA, "eval_path": None, "select": "all", "filter_cleared": True, "exclude_games": HELDOUT,
    "images": "placeholder", "max_seq_len": MAX_SEQ_LEN, "overlong": "window", "window_head": 8192, "mask_mode": "auto",
    "template_kwargs": {"enable_thinking": True}, "row_template_kwargs": True,
    "examples_cache": f"{OUT_DIR}/examples_cache.jsonl",
    "lora_r": 16, "lora_alpha": 32, "lora_dropout": 0.0,
    "lora_targets": ["q_proj", "k_proj", "v_proj", "o_proj", "in_proj_qkv", "in_proj_z", "out_proj"],
    "lora_exclude": ["visual", "vision", "(^|\\.)mtp\\.", "lm_head", "embed", "indexer", "ple",
                     "hyper_connection", "shared_expert"],
    "epochs": EPOCHS, "max_steps": MAX_STEPS, "lr": 1e-4, "warmup_ratio": 0.05, "batch_size": 1, "grad_accum": 4,
    "max_grad_norm": 1.0, "gradient_checkpointing": True, "loss_chunk": 2048, "log_every": 1,
    "save_every": SAVE_EVERY, "resume": True, "max_train_hours": MAX_TRAIN_HOURS, "oom_skip_limit": 20,
    "eval_max_examples": 4, "gen_samples": 0, "reload_check": False, "min_loss_drop": 0.0, "save_merged": False,
    "output_dir": OUT_DIR,
}
CFG = f"{CODE_DIR}/Alignment/colab_run.json"
open(CFG, "w").write(json.dumps(cfg, indent=1))
print(CFG)
!cd {CODE_DIR} && python tests/unit/test_alignment_nvfp4.py | grep -E "^(FAIL|CHECKS|HARD|INFO)"

In [ ]:
# 7. probe: loads the real model (~10-20 min) and trains 2 steps per length; prints peak VRAM and tokens/s.
#    Send this output before training. "ok": false or an OOM -> raise OFFLOAD_EXPERT_LAYERS in cell 1, rerun 6 and 7.
!cd {CODE_DIR}/Alignment && PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True TOKENIZERS_PARALLELISM=false \
    python -u long_probe.py --config {CFG} --lengths {PROBE_LENGTHS} --steps 2 2>&1 | tee {OUT_DIR}/probe.log
!free -g | head -2

In [ ]:
# 8. train (resumes from OUT_DIR/checkpoint after a disconnect). Keep this tab open.
#    A step that runs out of GPU memory is skipped and logged; a crash restarts from the last checkpoint.
import os, subprocess, time
env = {**os.environ, "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True", "TOKENIZERS_PARALLELISM": "false"}
for attempt in range(RETRIES + 1):
    with open(f"{OUT_DIR}/train.log", "a") as log:
        p = subprocess.Popen(["python", "-u", "lora_sft.py", "--config", CFG], cwd=f"{CODE_DIR}/Alignment", env=env,
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            print(line, end="")
            log.write(line)
            log.flush()
        code = p.wait()
    if code == 0:
        break
    msg = f"=== exit code {code}: restart {attempt + 1} of {RETRIES} from the checkpoint ==="
    print(msg)
    open(f"{OUT_DIR}/train.log", "a").write(msg + "\n")
    time.sleep(30)
!ls -la {OUT_DIR} {OUT_DIR}/adapter

In [ ]:
# 9. adapter -> the bf16 shards that hold the LoRA targets (the NVFP4 experts are untouched; ~15 GB to Drive).
#    Works on a fresh runtime: a CPU runtime with High-RAM is enough. Run cells 1 and 3 first, then this one.
#    Downloads only the config files and the shards it needs, not the whole 135 GB.
import json, os
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"
!pip install -q "safetensors==0.8.0" "huggingface_hub[hf_xet]==2.2.0"
!hf download {MODEL_REPO} --revision {MODEL_REV} --local-dir {MODEL_DIR} --exclude "*.safetensors"
from safetensors import safe_open
with safe_open(f"{OUT_DIR}/adapter/adapter_model.safetensors", "pt") as fh:
    akeys = list(fh.keys())
targets = {k.split(".lora_A.")[0].removeprefix("base_model.model.") + ".weight" for k in akeys if ".lora_A." in k}
wmap = json.load(open(f"{MODEL_DIR}/model.safetensors.index.json"))["weight_map"]
SHARDS = " ".join(sorted({wmap[t] for t in targets if t in wmap}))
print(len(targets), "targets; shards:", SHARDS, "| not found in the checkpoint:", sum(t not in wmap for t in targets))
!hf download {MODEL_REPO} --revision {MODEL_REV} --local-dir {MODEL_DIR} {SHARDS}
!cd {CODE_DIR}/Alignment && python -u merge_lora.py --base {MODEL_DIR} --adapter {OUT_DIR}/adapter \
    --out {OUT_DIR}/nvfp4_overlay --touched-only 2>&1 | tee {OUT_DIR}/merge.log
!ls -la {OUT_DIR}/nvfp4_overlay
from google.colab import drive; drive.flush_and_unmount()   # waits until the overlay is fully on Drive